# 09. 모델 계열 변경: LR → XGBoost (H8)

## 배경
- LR 계열은 피처를 바꿔도 LB 0.270~0.274에서 정체 (05, 07 E, 07 F)
- 07 확인 1: stats를 넣으면 LR은 test에서 변이 많은 클래스로 쏠림 → 선형 합산 구조가 원인일 가능성(로지스틱회귀라서 점수가 정체이지 않을까)

## 가설
- H8: 트리 모델은 피처를 합산하지 않고 분기로 판단하므로, 같은 피처(any + stats)에서 LB가 LR보다 높다
- H0: 모델을 바꿔도 LB는 0.27 근처에 머문다

## 검증 방법
- 피처: 07 B 조합(any + stats), 모델만 교체
- 하이퍼파라미터: 02와 같은 제공 베이스라인 설정 (튜닝은 10에서)
- 클래스 불균형: balanced sample_weight (LR의 class_weight='balanced'와 같은 역할)
- 판단 기준: LB가 05 최고값 0.2739보다 오르면 H8 채택

#### 데이터 준비와 XGBoost 학습
- 피처: `mutation_features()`의 any + stats (07 B 조합과 동일) → 모델만 바꾼 효과를 보기 위해 피처는 고정
- `BalancedXGB`: XGBoost에는 `class_weight` 옵션이 없어서, `fit`할 때 `compute_sample_weight('balanced')`를 자동으로 넣는 래퍼를 만듦
  - LR에서 쓴 `class_weight='balanced'`와 같은 역할 → LR과 같은 조건에서 비교하기 위함
  - 가중치는 `fit`에 들어온 y(= fold 학습 부분)로만 계산되므로 Data Leakage 아님
- 하이퍼파라미터: 02와 같은 제공 베이스라인 설정 (n_estimators=100, lr=0.1, max_depth=6)
  - 설정까지 바꾸면 두 가지를 동시에 바꾸는 것이므로, 튜닝은 10에서 따로 진행
- `run_cv`, `save_submission`: 팀 공통 함수 그대로 사용 → 다른 실험과 CV 점수를 직접 비교 가능

In [ ]:
# 09. H8: 트리 모델(XGBoost)은 LR보다 test 분포 차이에 덜 민감하다
import sys; sys.path.append('..')
import pandas as pd, xgboost as xgb
from sklearn.utils.class_weight import compute_sample_weight
from src.config import SEED
from src.data import load_data, get_gene_cols
from src.features import mutation_features
from src.validation import seed_everything, encode_target, run_cv, save_submission

seed_everything()
train, test, submission = load_data()
genes = get_gene_cols(train)
y, le = encode_target(train)
F_tr, F_te = mutation_features(train, genes), mutation_features(test, genes)

# XGBoost는 class_weight가 없어서, fit할 때 balanced 가중치를 자동으로 넣는 래퍼를 둠
class BalancedXGB(xgb.XGBClassifier):
    def fit(self, X, y, **kw):
        return super().fit(X, y, sample_weight=compute_sample_weight('balanced', y), **kw)

make_xgb = lambda: BalancedXGB(n_estimators=100, learning_rate=0.1, max_depth=6,
                            tree_method='hist', eval_metric='mlogloss',
                            random_state=SEED, n_jobs=-1)

X = pd.concat([F_tr['any'], F_tr['stats']], axis=1)
X_test = pd.concat([F_te['any'], F_te['stats']], axis=1)
r = run_cv(make_xgb, X, y, X_test)
save_submission(submission, r['test_proba'], le, name='xgb_any_stats', cv_score=r['cv_mean'])

[fold 0] Macro F1 = 0.3963
[fold 1] Macro F1 = 0.3928
[fold 2] Macro F1 = 0.3858
[fold 3] Macro F1 = 0.3809
[fold 4] Macro F1 = 0.3837
CV Macro F1 = 0.3879 ± 0.0058 (OOF 0.3889)
저장 완료: C:\Users\USER\Desktop\hackathon\submissions\1110_xgb_any_stats_0.388.csv


WindowsPath('C:/Users/USER/Desktop/hackathon/submissions/1110_xgb_any_stats_0.388.csv')

## 결과

| 모델 | 피처 | CV (± std) | LB | 05 최고 LB 대비 |
| :--- | :--- | :-: | :-: | :-: |
| LR (05) | any | 0.3818 ± 0.0064 | 0.2739 | — |
| LR (07 E) | any + trunc + stats | 0.4188 ± 0.0173 | 0.2706 | −0.003 |
| **XGBoost (09)** | any + stats | **0.3879 ± 0.0058** | **0.2995** | **+0.026** |

## 결론
- **H8 채택**: 같은 피처에서 모델만 바꿨는데 LB +0.026 → LR 계열이 넘지 못하던 0.27 벽을 처음 돌파
- CV−LB 차이: LR 0.108~0.148 → XGBoost 0.088로 감소
  - LR은 피처를 더하는 구조라 test의 변이 수 증가에 그대로 끌려감 (07 확인 1: LUAD 3.68배 쏠림)
  - 트리는 "변이 수 > 기준값?" 같은 분기로 판단해서, train 범위를 넘는 값도 마지막 구간에 머묾 → 쏠림이 덜함
- CV는 07 E(0.4188)보다 낮지만 LB는 높음 → 이 대회에서는 CV 평균보다 std와 test 안정성이 중요 (재확인)
- std 0.0058은 지금까지 중 가장 작음 → fold마다 고르게 맞힘

## 다음
- 10: 기본 설정은 규제가 없으므로 하이퍼파라미터 탐색